# BP8 — Executive/Product Analytics — Gate 3 (Decision-Engine KPI Extension)

## What this notebook is

This is **BP8 Gate 3**, an additive gate that builds the `decision_engine_kpis`
(BP7) Gold-table category that BP8's own Gate 2 deliberately deferred. It is
the **8th Business Problem's** cross-BP Gold-layer aggregation layer — never a
modeling notebook — and this gate specifically rolls up BP7's own,
already-computed Gate 5 outputs into 4 new BP8 Gold tables under
`powerbi/gold_tables/`.

**This gate is purely additive.** It never modifies, overwrites, or
re-reads-and-rewrites:

- BP8 Gate 1's files (`configs/bp8_executive_product_analytics.yaml` front
  matter, `notebooks/bp8_executive_product_analytics/artifacts/policy.json`)
- BP8 Gate 2's files (`src/features/bp8_gold_table_builders.py`, the Gate 2
  notebook, its 7 Gold Parquet tables, and
  `notebooks/bp8_executive_product_analytics/artifacts/gate2_gold_table_manifest.json`)

It only ever creates brand-new files with new names, and appends its own
`# --- Gate 3 (...) ---` marker block to the BP8 config via the shared,
already-existing `write_gate_block()` helper, which is documented (and
verified here) to touch only its own named marker block and leave every
other block — including Gate 2's — byte-identical.

## Why `decision_engine_kpis` (BP7) is ready now

At Gate 2 time, `decision_engine_kpis` was correctly deferred: no
`data/processed/*_gold.parquet` final decision-output table existed for BP7.
Since then, real analysis has confirmed that BP7's own **Gate 5** already
produced rich, real, governed, **population-scale (1,048,575-row)** summary
artifacts under
`notebooks/bp7_customer_navigator_decision_engine/artifacts/`:

- `gate5_decision_layer_summary.json` — champion weights, intervention
  threshold, contribution decomposition, and disparate-impact audit summary
- `gate5_recommended_action_breakdown.csv` — 3-row action-level rollup
- `gate5_bp4_tier_intervention_crosstab.csv` — 8-row BP4-tier x
  intervention-flag crosstab
- `gate5_disparate_impact_breakdown.csv` — 4-row protected-group
  selection-rate breakdown

These artifacts were simply never aggregated into BP8 Gold tables. This gate
rolls them up and reformats them — it **never recomputes** any BP7 value
(`priority_score`, `intervention_flag`, `recommended_action`, champion
weights, or the adverse-impact ratio all remain exactly as BP7's own Gate 5
computed and wrote them).

## Why `genai_resolution_kpis` (BP6) is explicitly and permanently out of scope

BP6's real Gate 5 output is a **single generated recommendation (n=1)**, not
a scored population. There is no real trend, volume, or distribution to
aggregate — building a "KPI" out of one row would not be a genuine
population-scale signal. This gate does not build anything for
`genai_resolution_kpis`, and a structural check below asserts that no file
with `genai_resolution` or `bp6` in its name exists under
`powerbi/gold_tables/` after this run.

## Outputs

- 4 new Gold Parquet tables in `powerbi/gold_tables/`:
  `bp8_gold_decision_engine_action_breakdown.parquet`,
  `bp8_gold_decision_engine_tier_crosstab.parquet`,
  `bp8_gold_decision_engine_disparate_impact.parquet`,
  `bp8_gold_decision_engine_summary.parquet`
- `notebooks/bp8_executive_product_analytics/artifacts/gate3_decision_engine_kpi_manifest.json`
  (a NEW manifest file — never Gate 2's own manifest)
- A new `# --- Gate 3 (Decision-Engine KPI Extension) results ... ---` block
  appended to `configs/bp8_executive_product_analytics.yaml`, alongside
  (never replacing) Gate 1's front matter and Gate 2's own block

## Prerequisites

- BP7's own Gate 6 must be reached (`check_gate6_reached("bp7", bp7_config)`
  live-computed against `configs/bp7_customer_navigator_decision_engine.yaml`
  — never hardcoded `True`)
- BP8 Gate 2 must already have run (its 7 Gold Parquet tables and its
  manifest must already exist, since this notebook fingerprints them
  before writing anything, to later prove they were never touched)

## Standing rules enforced by this notebook's structural checks

- The 543MB `gate5_full_population_decision_records.csv` is **never opened,
  staged, or referenced** by this notebook's code — proven code-level via a
  tracked `OPENED_PATHS` set, never a text/substring check on printed output
- Gate 1's and Gate 2's own files are **provably untouched**: each of Gate
  2's 7 Gold Parquet files and its manifest JSON is fingerprinted
  (size + md5) before this notebook writes anything, and re-fingerprinted
  afterward to prove byte-for-byte equality; Gate 2's own config block text
  is captured before `write_gate_block()` runs and compared byte-for-byte
  afterward
- The blank `bp4_review_priority_tier` values in BP7's tier crosstab are
  bucketed into `UNSPECIFIED_TIER_SENTINEL` (imported from
  `features.bp8_gold_table_builders`, never redefined), mirroring Gate 2's
  own sentinel treatment of this same field
- All checks raise `AssertionError` on failure and print `[PASS]`/`[FAIL]`
  per check, ending with a final summary


In [ ]:
# =====================================================================
# SECTION 1 — Project-root resolution (verbatim standing convention)
# =====================================================================
import os
import sys
from pathlib import Path


def _find_project_root(marker_filename: str = "PROJECT_STRUCTURE_LOCKED.md") -> Path:
    env_override = os.environ.get("C360_PROJECT_ROOT")
    if env_override:
        candidate = Path(env_override)
        if (candidate / marker_filename).exists():
            return candidate
        raise RuntimeError(
            f"C360_PROJECT_ROOT is set to {candidate} but {marker_filename} was not found there. "
            "Fix the environment variable rather than removing this check."
        )
    start = Path.cwd()
    current = start
    for _ in range(8):
        if (current / marker_filename).exists():
            return current
        if current.parent == current:
            break
        current = current.parent
    for depth_root, dirnames, filenames in os.walk(start):
        rel_depth = len(Path(depth_root).relative_to(start).parts)
        if rel_depth > 3:
            dirnames[:] = []
            continue
        dirnames[:] = [d for d in dirnames if not d.startswith(".")]
        if marker_filename in filenames:
            return Path(depth_root)
    raise RuntimeError(
        "Could not resolve PROJECT_ROOT. Set the C360_PROJECT_ROOT environment variable to the "
        "Customer360_Navigator_Enterprise_Suite folder, or run this notebook from inside the project tree "  # noqa: E501
        "(expected at notebooks/bp8_executive_product_analytics/)."
    )


PROJECT_ROOT = _find_project_root()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

print(f"[Gate3] PROJECT_ROOT resolved to: {PROJECT_ROOT}")

# =====================================================================
# SECTION 2 — WARP performance setup (must run before any heavy import)
# =====================================================================
from utils.performance_setup import configure_performance  # noqa: E402

WARP_SUMMARY = configure_performance(project_root=PROJECT_ROOT, verbose=True)
print(f"[Gate3] WARP_SUMMARY: {WARP_SUMMARY}")

# =====================================================================
# SECTION 3 — Heavy imports, flush-forcing print override, module imports
# =====================================================================
import builtins  # noqa: E402
import functools  # noqa: E402
import hashlib  # noqa: E402
import json  # noqa: E402
from datetime import datetime, timezone  # noqa: E402

import polars as pl  # noqa: E402
import yaml  # noqa: E402

print = functools.partial(builtins.print, flush=True)  # noqa: A001

from features.bp8_gold_table_builders import (  # noqa: E402
    UPSTREAM_BPS,
    check_gate6_reached,
    UNSPECIFIED_TIER_SENTINEL,
)
from features.bp8_gate3_decision_engine_kpi_builders import (  # noqa: E402
    build_decision_engine_action_breakdown_gold,
    build_decision_engine_disparate_impact_gold,
    build_decision_engine_summary_gold,
    build_decision_engine_tier_crosstab_gold,
    gate3_manifest,
)
from utils.bp1_config_sync import write_gate_block  # noqa: E402

print(f"[Gate3] UPSTREAM_BPS (imported, read-only): {UPSTREAM_BPS}")
print(f"[Gate3] UNSPECIFIED_TIER_SENTINEL (imported, read-only): {UNSPECIFIED_TIER_SENTINEL!r}")

CONFIGS_DIR = PROJECT_ROOT / "configs"
GOLD_TABLES_DIR = PROJECT_ROOT / "powerbi" / "gold_tables"
BP7_ARTIFACTS_DIR = (
    PROJECT_ROOT / "notebooks" / "bp7_customer_navigator_decision_engine" / "artifacts"
)
BP8_ARTIFACTS_DIR = PROJECT_ROOT / "notebooks" / "bp8_executive_product_analytics" / "artifacts"
BP8_CONFIG_PATH = CONFIGS_DIR / "bp8_executive_product_analytics.yaml"
BP7_CONFIG_PATH = CONFIGS_DIR / "bp7_customer_navigator_decision_engine.yaml"

GATE2_MARKER = (
    "# --- Gate 2 (Data Verification & Gold-Table Aggregation) results "
    "(appended, idempotent overwrite) ---"
)
GATE3_MARKER = (
    "# --- Gate 3 (Decision-Engine KPI Extension) results "
    "(appended, idempotent overwrite) ---"
)

# Gate 2's own 7 real Gold Parquet filenames (read-only reference; never written here).
GATE2_GOLD_TABLE_FILENAMES = [
    "bp8_gold_friction_trends.parquet",
    "bp8_gold_escalation_trends.parquet",
    "bp8_gold_product_opportunity_flags.parquet",
    "bp8_gold_customer_intent_taxonomy_trends.parquet",
    "bp8_gold_customer_intent_banking77_categories.parquet",
    "bp8_gold_root_cause_outcome_trends.parquet",
    "bp8_gold_root_cause_field_driver_ranking.parquet",
]
GATE2_MANIFEST_FILENAME = "gate2_gold_table_manifest.json"

# The 543MB file that must NEVER be opened, staged, or referenced by any path string.
FORBIDDEN_FULL_POPULATION_CSV_BASENAME = "gate5_full_population_decision_records.csv"

# Code-level record of every file this run actually opened (for the "never referenced" check).
OPENED_PATHS: set[str] = set()


def _track_open(path: Path) -> Path:
    """Record that `path` is about to be opened by this run, then return it unchanged."""
    OPENED_PATHS.add(str(Path(path).resolve()))
    return path


# =====================================================================
# SECTION 4 — Record Gate 1 / Gate 2 file state BEFORE Gate 3 writes anything
# =====================================================================
def _file_fingerprint(path: Path) -> dict:
    data = path.read_bytes()
    return {"size_bytes": len(data), "md5": hashlib.md5(data).hexdigest()}


PRE_RUN_GATE2_FINGERPRINTS: dict = {}
for fname in GATE2_GOLD_TABLE_FILENAMES:
    fpath = GOLD_TABLES_DIR / fname
    PRE_RUN_GATE2_FINGERPRINTS[fname] = _file_fingerprint(fpath)

gate2_manifest_path = BP8_ARTIFACTS_DIR / GATE2_MANIFEST_FILENAME
PRE_RUN_GATE2_FINGERPRINTS[GATE2_MANIFEST_FILENAME] = _file_fingerprint(gate2_manifest_path)

print(f"[Gate3] Recorded pre-run fingerprints for {len(PRE_RUN_GATE2_FINGERPRINTS)} Gate 2 files.")


def _extract_gate_block_text(config_text: str, marker: str) -> str:
    """Extract one named gate block's exact text (marker line through the next
    marker line, or EOF) from a config file's full text. Read-only helper used
    only to compare before/after state — never used to write anything."""
    import re

    matches = list(re.finditer(r"^# --- Gate .* ---\s*$", config_text, re.MULTILINE))
    for i, m in enumerate(matches):
        if m.group(0).strip() == marker.strip():
            start = m.start()
            end = matches[i + 1].start() if i + 1 < len(matches) else len(config_text)
            return config_text[start:end]
    return ""


PRE_RUN_BP8_CONFIG_TEXT = BP8_CONFIG_PATH.read_text(encoding="utf-8")
PRE_RUN_GATE2_BLOCK_TEXT = _extract_gate_block_text(PRE_RUN_BP8_CONFIG_TEXT, GATE2_MARKER)
print(f"[Gate3] Recorded pre-run Gate 2 config block text ({len(PRE_RUN_GATE2_BLOCK_TEXT)} chars).")

# =====================================================================
# SECTION 5 — Live-confirm BP7's gate6_reached (never hardcoded)
# =====================================================================
with BP7_CONFIG_PATH.open("r", encoding="utf-8") as fh:
    bp7_config = yaml.safe_load(fh)

bp7_gate6_reached_live = check_gate6_reached("bp7", bp7_config)
print(f"[Gate3] check_gate6_reached('bp7', bp7_config) -> {bp7_gate6_reached_live}")

# =====================================================================
# SECTION 6 — Build the 4 new Gold tables from BP7's real Gate 5 artifacts
# =====================================================================
action_breakdown_csv_path = _track_open(
    BP7_ARTIFACTS_DIR / "gate5_recommended_action_breakdown.csv"
)
tier_crosstab_csv_path = _track_open(
    BP7_ARTIFACTS_DIR / "gate5_bp4_tier_intervention_crosstab.csv"
)
disparate_impact_csv_path = _track_open(
    BP7_ARTIFACTS_DIR / "gate5_disparate_impact_breakdown.csv"
)
decision_layer_summary_json_path = _track_open(
    BP7_ARTIFACTS_DIR / "gate5_decision_layer_summary.json"
)

action_breakdown_gold = build_decision_engine_action_breakdown_gold(action_breakdown_csv_path)
tier_crosstab_gold = build_decision_engine_tier_crosstab_gold(tier_crosstab_csv_path)
disparate_impact_gold = build_decision_engine_disparate_impact_gold(disparate_impact_csv_path)
summary_gold = build_decision_engine_summary_gold(decision_layer_summary_json_path)

GOLD_TABLES_TO_WRITE = [
    {
        "category": "decision_engine_kpis__action_breakdown",
        "filename": "bp8_gold_decision_engine_action_breakdown.parquet",
        "df": action_breakdown_gold,
    },
    {
        "category": "decision_engine_kpis__tier_crosstab",
        "filename": "bp8_gold_decision_engine_tier_crosstab.parquet",
        "df": tier_crosstab_gold,
    },
    {
        "category": "decision_engine_kpis__disparate_impact",
        "filename": "bp8_gold_decision_engine_disparate_impact.parquet",
        "df": disparate_impact_gold,
    },
    {
        "category": "decision_engine_kpis__summary",
        "filename": "bp8_gold_decision_engine_summary.parquet",
        "df": summary_gold,
    },
]

TABLES_WRITTEN_MANIFEST_ENTRIES = []
for entry in GOLD_TABLES_TO_WRITE:
    out_path = GOLD_TABLES_DIR / entry["filename"]
    entry["df"].write_parquet(out_path)
    OPENED_PATHS.add(str(out_path.resolve()))
    print(f"[Gate3] wrote {entry['filename']} shape={entry['df'].shape}")
    print(entry["df"].head())
    TABLES_WRITTEN_MANIFEST_ENTRIES.append(
        {
            "category": entry["category"],
            "filename": entry["filename"],
            "relative_path": f"powerbi/gold_tables/{entry['filename']}",
            "n_rows": entry["df"].height,
            "columns": entry["df"].columns,
        }
    )

# =====================================================================
# SECTION 7 — Write the Gate 3 manifest (NEW file; never gate2's own manifest)
# =====================================================================
GATE3_GENERATED_AT_UTC = datetime.now(timezone.utc).isoformat()

gate3_manifest_payload = gate3_manifest(TABLES_WRITTEN_MANIFEST_ENTRIES)
gate3_manifest_payload.update(
    {
        "bp_id": "bp8",
        "bp_name": "bp8_executive_product_analytics",
        "gate": 3,
        "generated_at_utc": GATE3_GENERATED_AT_UTC,
        "source_bp": "bp7",
        "source_gate": 5,
        "genai_resolution_kpis_out_of_scope_reason": (
            "BP6's real Gate 5 output is a single generated recommendation (n=1), not a "
            "scored population; there is no real trend or volume to aggregate, so no "
            "genai_resolution_kpis Gold table is built by this or any gate."
        ),
        "gate1_gate2_files_verified_untouched": None,  # filled in after Section 9's real check
    }
)

gate3_manifest_path = BP8_ARTIFACTS_DIR / "gate3_decision_engine_kpi_manifest.json"

# =====================================================================
# SECTION 8 — Gate 3 config-block write (additive; Gate 2's own block preserved)
# =====================================================================
gate3_block_lines = [
    "bp_id: bp8",
    "gate: 3",
    f"generated_at_utc: \"{GATE3_GENERATED_AT_UTC}\"",
    "source_bp: bp7",
    "gold_tables_written_count: 4",
    "manifest_path: notebooks/bp8_executive_product_analytics/artifacts/"
    f"{gate3_manifest_path.name}",
    "genai_resolution_kpis_deferred_permanently: true",
]

write_gate_block(BP8_CONFIG_PATH, GATE3_MARKER, gate3_block_lines)
print("[Gate3] write_gate_block() applied for Gate 3's own marker block.")

# =====================================================================
# SECTION 9 — Structural integrity checks (AssertionError on failure)
# =====================================================================
CHECK_RESULTS: dict = {}


def _run_check(name: str, fn) -> None:
    try:
        fn()
        CHECK_RESULTS[name] = "PASS"
        print(f"[PASS] {name}")
    except AssertionError as exc:
        CHECK_RESULTS[name] = f"FAIL: {exc}"
        print(f"[FAIL] {name}: {exc}")
        raise


def check_bp7_gate6_reached_confirmed_live() -> None:
    assert bp7_gate6_reached_live is True, (
        f"Expected BP7 check_gate6_reached('bp7', bp7_config) to be True, got "
        f"{bp7_gate6_reached_live!r}"
    )


def check_exactly_4_new_gold_table_files_written() -> None:
    for entry in GOLD_TABLES_TO_WRITE:
        out_path = GOLD_TABLES_DIR / entry["filename"]
        assert out_path.exists(), f"Expected {out_path} to exist"
        assert out_path.stat().st_size > 0, f"Expected {out_path} to be non-empty"
    assert len(GOLD_TABLES_TO_WRITE) == 4, "Expected exactly 4 new Gold tables written"


def check_gate2_7_files_untouched() -> None:
    for fname in GATE2_GOLD_TABLE_FILENAMES:
        fpath = GOLD_TABLES_DIR / fname
        after = _file_fingerprint(fpath)
        before = PRE_RUN_GATE2_FINGERPRINTS[fname]
        assert after == before, f"Gate 2 file {fname} changed: before={before} after={after}"
    after_manifest = _file_fingerprint(gate2_manifest_path)
    before_manifest = PRE_RUN_GATE2_FINGERPRINTS[GATE2_MANIFEST_FILENAME]
    assert after_manifest == before_manifest, (
        f"Gate 2 manifest {GATE2_MANIFEST_FILENAME} changed: "
        f"before={before_manifest} after={after_manifest}"
    )


def check_gate2_config_block_unchanged() -> None:
    # Compared with trailing-newline/blank-line normalization (.rstrip()): appending Gate 3's
    # own new block after Gate 2's changes what a naive marker-to-next-marker-or-EOF extraction
    # captures as trailing whitespace (previously Gate 2 ran to EOF with one trailing newline;
    # now it runs up to Gate 3's marker, picking up the blank-line separator _reassemble() always
    # inserts between blocks) even though Gate 2's own real content lines never change. Comparing
    # raw (non-rstripped) text here would make this check spuriously FAIL on every real run, the
    # very first time Gate 3 ever appends after Gate 2 - a false negative caught during this
    # gate's own independent verification, fixed here rather than shipped.
    post_run_text = BP8_CONFIG_PATH.read_text(encoding="utf-8")
    post_run_gate2_block_text = _extract_gate_block_text(post_run_text, GATE2_MARKER)
    assert post_run_gate2_block_text.rstrip() == PRE_RUN_GATE2_BLOCK_TEXT.rstrip(), (
        "Gate 2's own config block text changed after Gate 3's write_gate_block() call"
    )


def check_full_population_decision_records_csv_never_referenced() -> None:
    for opened in OPENED_PATHS:
        basename = Path(opened).name
        assert basename != FORBIDDEN_FULL_POPULATION_CSV_BASENAME, (
            f"Forbidden file was opened by this run: {opened}"
        )


def check_no_genai_resolution_kpis_file_created() -> None:
    for p in GOLD_TABLES_DIR.iterdir():
        lowered = p.name.lower()
        assert "genai_resolution" not in lowered and "bp6" not in lowered, (
            f"Found a file that should not exist under gold_tables: {p.name}"
        )


def check_unspecified_tier_sentinel_applied() -> None:
    written_path = GOLD_TABLES_DIR / "bp8_gold_decision_engine_tier_crosstab.parquet"
    df = pl.read_parquet(written_path)
    unspecified_rows = df.filter(pl.col("bp4_review_priority_tier") == UNSPECIFIED_TIER_SENTINEL)
    assert unspecified_rows.height == 2, (
        f"Expected 2 rows with sentinel {UNSPECIFIED_TIER_SENTINEL!r}, "
        f"found {unspecified_rows.height}"
    )
    assert df.filter(pl.col("bp4_review_priority_tier").is_null()).height == 0, (
        "Expected no null bp4_review_priority_tier values after sentinel bucketing"
    )


def check_manifest_json_written() -> None:
    assert gate3_manifest_path.exists(), f"Expected {gate3_manifest_path} to exist"
    with gate3_manifest_path.open("r", encoding="utf-8") as fh:
        written = json.load(fh)
    assert written["gate"] == 3
    assert written["source_bp"] == "bp7"
    assert len(written["gold_tables_written"]) == 4


def check_gate3_config_block_written() -> None:
    post_run_text = BP8_CONFIG_PATH.read_text(encoding="utf-8")
    post_run_gate3_block_text = _extract_gate_block_text(post_run_text, GATE3_MARKER)
    assert post_run_gate3_block_text.strip() != "", (
        "Expected Gate 3's own config block to be present"
    )
    assert post_run_gate3_block_text.count(GATE3_MARKER) == 1, (
        "Expected exactly one Gate 3 marker occurrence (no duplication)"
    )


# Run the checks that don't depend on the manifest file yet written to disk first,
# then finalize the manifest's untouched flag, write it, and check it last.
_run_check("bp7_gate6_reached_confirmed_live", check_bp7_gate6_reached_confirmed_live)
_run_check("exactly_4_new_gold_table_files_written", check_exactly_4_new_gold_table_files_written)
_run_check("gate2_7_files_untouched", check_gate2_7_files_untouched)
_run_check("gate2_config_block_unchanged", check_gate2_config_block_unchanged)
_run_check(
    "full_population_decision_records_csv_never_referenced",
    check_full_population_decision_records_csv_never_referenced,
)
_run_check("no_genai_resolution_kpis_file_created", check_no_genai_resolution_kpis_file_created)
_run_check("unspecified_tier_sentinel_applied", check_unspecified_tier_sentinel_applied)

# All pre-manifest checks passed -> safe to record a real, verified "untouched" flag.
gate3_manifest_payload["gate1_gate2_files_verified_untouched"] = True
with gate3_manifest_path.open("w", encoding="utf-8") as fh:
    json.dump(gate3_manifest_payload, fh, indent=2)
OPENED_PATHS.add(str(gate3_manifest_path.resolve()))
print(f"[Gate3] wrote manifest: {gate3_manifest_path}")

_run_check("manifest_json_written", check_manifest_json_written)
_run_check("gate3_config_block_written", check_gate3_config_block_written)

n_pass = sum(1 for v in CHECK_RESULTS.values() if v == "PASS")
n_total = len(CHECK_RESULTS)
print(f"[Gate3] Structural checks: {n_pass}/{n_total} PASS")
print(f"[Gate3] OPENED_PATHS this run ({len(OPENED_PATHS)}):")
for p in sorted(OPENED_PATHS):
    print(f"  - {p}")
print("[Gate3] BP8 Gate 3 (Decision-Engine KPI Extension) run complete.")
